In [1]:
import xmlrpc.client

# =========================================================================
# 1. ENVIRONMENT INITIALIZATION & PROXY CONNECTION
# =========================================================================
URL = "http://localhost:8069"
DB = "m.nushath"       
USERNAME = "user@company.com"  
API_KEY = "user"              

print("Initializing XML-RPC connections for Delete Operations Demo...")
common = xmlrpc.client.ServerProxy(f'{URL}/xmlrpc/2/common', allow_none=True)
models = xmlrpc.client.ServerProxy(f'{URL}/xmlrpc/2/object', allow_none=True)

uid = common.authenticate(DB, USERNAME, API_KEY, {})
if not uid:
    print("[CRITICAL] Authentication failed!")
    exit()

print(f"[SUCCESS] Connected securely as User ID: {uid}\n" + "="*70)

# Fetch context parameters
user_data = models.execute_kw(DB, uid, API_KEY, 'res.users', 'read', [[uid]], {'fields': ['company_id', 'company_ids']})
current_company_id = user_data[0]['company_id'][0]
allowed_company_ids = user_data[0]['company_ids']
base_context = {'company_id': current_company_id, 'allowed_company_ids': allowed_company_ids}

Initializing XML-RPC connections for Delete Operations Demo...
[SUCCESS] Connected securely as User ID: 5


In [4]:
# =========================================================================
# GENERATING TRANSACTING MOCK TARGETS (Ensures code executes reliably)
# =========================================================================
print("[Setup] Provisioning mock test elements...")

# Generate a temporary mock contact for the Hard Delete demo
hard_target_id = models.execute_kw(DB, uid, API_KEY, 'res.partner', 'create', [{
    'name': 'API Temp Delete Target - Purge Testing Ltd',
    'company_id': current_company_id
}])

# Generate a temporary mock contact for the Soft Delete/Archive demo
soft_target_id = models.execute_kw(DB, uid, API_KEY, 'res.partner', 'create', [{
    'name': 'API Temp Archive Target - Record Retention Inc',
    'company_id': current_company_id
}])

print(f" -> Temporary Hard-Delete Target ID: {hard_target_id}")
print(f" -> Temporary Soft-Delete Target ID: {soft_target_id}\n" + "-"*50)

[Setup] Provisioning mock test elements...
 -> Temporary Hard-Delete Target ID: 22
 -> Temporary Soft-Delete Target ID: 23
--------------------------------------------------


In [5]:
# =========================================================================
# DEMO 1: HARD DELETE EXECUTION (unlink)
# Documentation Link: Recipe #1
# =========================================================================
print("[DEMO 1] Executing Permanent Database Unlink (Hard Delete)...")

try:
    # Pass the list of IDs into the array parameter mapping slot
    unlink_success = models.execute_kw(DB, uid, API_KEY, 'res.partner', 'unlink', 
        [[hard_target_id]],
        {'context': base_context}
    )
    print(f" -> Method output data type: {type(unlink_success)} (Returns Boolean)")
    print(f" -> [SUCCESS] Record permanently removed from DB: {unlink_success}")
except Exception as e:
    print(f" -> Unlink Blocked: {e}")

[DEMO 1] Executing Permanent Database Unlink (Hard Delete)...
 -> Method output data type: <class 'bool'> (Returns Boolean)
 -> [SUCCESS] Record permanently removed from DB: True


In [6]:
# =========================================================================
# DEMO 2: SOFT DELETE EXECUTION (Archiving via active=False)
# Documentation Link: Recipe #2
# =========================================================================
print("\n" + "-"*50 + "\n[DEMO 2] Executing Active Flag Deactivation (Soft Delete)...")

archive_vals = {
    'active': False # Set to True to reverse this operation and unarchive the record
}

write_success = models.execute_kw(DB, uid, API_KEY, 'res.partner', 'write',
    [[soft_target_id], archive_vals],
    {'context': base_context}
)
print(f" -> [SUCCESS] Record deactivated/hidden from everyday visibility filters: {write_success}")


--------------------------------------------------
[DEMO 2] Executing Active Flag Deactivation (Soft Delete)...
 -> [SUCCESS] Record deactivated/hidden from everyday visibility filters: True


In [7]:
# =========================================================================
# DEMO 3: EXAMINING DB CONSTRAINT SAFETY BLOCKS
# Purpose: Demonstrate how Odoo protects data integrity from destructive unlinks
# =========================================================================
print("\n" + "-"*50 + "\n[DEMO 3] Simulating Integrity Constraint Safety Blocks...")

# Attempting to delete a core system user profile record (ID 5) which has audit ties
protected_record_id = 5 

try:
    print(f" -> Attempting to forcefully hard-delete protected User Record ID: {protected_record_id}...")
    blocked_output = models.execute_kw(DB, uid, API_KEY, 'res.users', 'unlink', [[protected_record_id]])
except Exception as e:
    print(f" -> [PROTECTED] Odoo engine successfully blocked the call. Safe crash payload captured:")
    print(f"    {str(e)[:120]}... [Truncated]")

print("\n" + "="*70 + "\n[FINISHED] Complete Delete CRUD reference operations completed successfully!")


--------------------------------------------------
[DEMO 3] Simulating Integrity Constraint Safety Blocks...
 -> Attempting to forcefully hard-delete protected User Record ID: 5...
 -> [PROTECTED] Odoo engine successfully blocked the call. Safe crash payload captured:
    <Fault 2: 'The operation cannot be completed: update or delete on table "res_users" violates RESTRICT setting of foreign... [Truncated]

[FINISHED] Complete Delete CRUD reference operations completed successfully!
